In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format("delta")\
    .load("abfss://silver@netflixproject28.dfs.core.windows.net/netflix_titles")


In [0]:
df = df.withColumn(
    "duration_minutes",
    when(col("type") == "Movie", col("duration_minutes")).otherwise(0))\
.withColumn(
    "duration_seasons",
    when(col("type") == "TV Show", col("duration_seasons")).otherwise(1)
)

In [0]:
df = df.withColumn("shorttitle", split(col('title'), ':')[0])


In [0]:
df = df.withColumn("rating", split(col('rating'), '-')[0])

In [0]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "type_flag",
    when(col("type") == "Movie", 1)
    .when(col("type") == "TV Show", 2)
    .otherwise(0)
)

In [0]:
df = df.withColumn("duaration_ranking", dense_rank().over(Window.orderBy(col("duration_minutes").desc())))



In [0]:
df_vis= df.groupBy(col("type")).agg(count("*").alias("total_count"))
display(df_vis)

Databricks visualization. Run in Databricks to view.

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
        .option("overwriteSchema", "true")\
    .save("abfss://silver@netflixproject28.dfs.core.windows.net/netflix_titles")